# Step 12 — Does Negative Correlation Learning generalize? (Higgs Small, Facebook Comments, Santander)

Notebook 11 found that NCL (`ncl.py`) beats our tuned XGBoost, published FT-Transformer, and
published MLP on California Housing — the strongest v4 result in the project — but that was
ONE dataset at ONE fixed architecture. This notebook asks the obvious next question: does it
generalize, or was that a California-Housing-specific fluke? Tested on the 3 datasets from
Step 6 (`benchmarks.py`, exact published splits) that are neither multiclass (not yet
supported by `ncl.py`) nor as large as Covertype: Higgs Small (binary, 62.8k rows),
Facebook Comments (regression, 157.6k rows), Santander (binary, 128k rows).

### Fixed config -- copied verbatim from notebook 07

`num_learners=16, hidden_dim=16, embed_dim=32, depth=1, feature_frac=0.7, dropout=0.1`, PLR
embedding `d_embedding=8, n_frequencies=16, sigma=0.1`, `lr=1e-3, weight_decay=1e-4,
batch_size=512`, `epochs=100, patience=10` -- notebook 07's exact `ENS_KW`/`EMB_KW`/
`V4_TRAIN_KW`, so results stay comparable to that notebook's numbers. Same capacity-parity
caveat as notebook 11: notebook 07's "v4 mean-pool + PLR" line used `meanpool_wide` (head
widened to match attention arms); this notebook's baseline is plain `meanpool` (small head,
same capacity as `meanpool_ncl`), so the wide-head numbers are shown as context only, not a
head-to-head comparison target.

### Lambda grid, informed by notebook 11's empirical finding

Notebook 11 (and `ncl.py`'s own toy self-test) found a smooth improvement from lambda=0 to
~1.0, then a sharp breakdown past that. This notebook sweeps `{0.0, 0.2, 0.4, 0.6, 0.8, 1.0}`
-- 6 values, no need to re-probe the instability zone past 1.0 again, already confirmed twice.

### Real GPU time estimate, grounded in notebook 07's actual recorded per-run times

At the wide-head PLR capacity (bigger than what this notebook uses), mean-pool+PLR took
~0.3-0.4 min/run on Higgs, ~1.1-1.7 min/run on Facebook Comments, ~0.6-0.7 min/run on
Santander. `meanpool_ncl`'s per-learner Python loop adds some overhead (confirmed ~20-50%
slower than plain independent training in notebook 11's CPU runs). With 7 configs (baseline +
6 lambdas) x 3 seeds x 3 datasets = 63 runs, and 3 parallel GPU workers (matching notebook
07/08's convention), total wall time should land roughly in the 25-45 minute range --
an estimate, not a guarantee; watch the progress log.

## 0. Setup

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd(); sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from joblib import Parallel, delayed

from benchmarks import BENCHMARKS, load_benchmark, published, higher_is_better
from training import get_device
from tuning import run_config

DEVICE = get_device()

# Knobs, overridable from the shell so a headless run needs no edits:
#   SRP_SUBSAMPLE=2000 SRP_SEEDS=1 SRP_EPOCHS=3 SRP_PATIENCE=2 SRP_KEYS=HI jupyter nbconvert ...
KEYS      = os.environ.get("SRP_KEYS", "HI,FB,SA").split(",")
LAMBDAS   = [float(v) for v in os.environ.get("SRP_LAMBDAS", "0.0,0.2,0.4,0.6,0.8,1.0").split(",")]
SEEDS     = tuple(range(int(os.environ.get("SRP_SEEDS", "3"))))
EPOCHS    = int(os.environ.get("SRP_EPOCHS", "100"))         # matches notebook 07's budget
PATIENCE  = int(os.environ.get("SRP_PATIENCE", "10"))
SUBSAMPLE = int(os.environ.get("SRP_SUBSAMPLE", "0")) or None

THREADS_PER_JOB = 2
N_JOBS = int(os.environ.get("SRP_JOBS", "0")) or (
    3 if DEVICE.type == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB)))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
STAMP = time.strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(RUN_DIR, f"12_progress_{STAMP}.log")
_latest = os.path.join(RUN_DIR, "12_progress.log")
if os.path.islink(_latest) or os.path.exists(_latest):
    os.remove(_latest)
os.symlink(os.path.basename(LOG_PATH), _latest)
def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")
def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

sns.set_theme(style="whitegrid", context="notebook")
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"
log(f"device: {DEVICE} ({gpu}) | keys={KEYS} | {N_JOBS} workers | "
    f"lambdas={LAMBDAS} seeds={len(SEEDS)} epochs={EPOCHS}")
if DEVICE.type != "cuda":
    print("NOTE: running on CPU -- FB and SA will be much slower here. Meant for the GPU server.")

## 1. Data and the fixed config (copied from notebook 07)

In [ ]:
DATA, ARRAYS = {}, {}
for k in KEYS:
    d = load_benchmark(k)
    arrays = d.arrays()
    if SUBSAMPLE:
        n = SUBSAMPLE
        arrays = {"Xtr": arrays["Xtr"][:n], "ytr": arrays["ytr"][:n],
                 "Xva": arrays["Xva"][:n // 2], "yva": arrays["yva"][:n // 2],
                 "Xte": arrays["Xte"][:n], "yte": arrays["yte"][:n]}
    DATA[k], ARRAYS[k] = d, arrays
    log(d.summary() + ("  [SUBSAMPLED]" if SUBSAMPLE else ""))

# Verbatim from notebook 07's ENS_KW / EMB_KW / V4_TRAIN_KW. head_hidden=embed_dim,
# head_depth=1 reproduces PredictionHead's default -- the "small head" plain meanpool arm,
# NOT meanpool_wide (see capacity-parity note above).
BASE_PARAMS = dict(num_learners=16, hidden_dim=16, embed_dim=32, depth=1, feature_frac=0.7,
                   dropout=0.1, lr=1e-3, weight_decay=1e-4, head_hidden=32, head_depth=1,
                   embedding_mode="periodic", d_embedding=8, n_frequencies=16, sigma=0.1)
print("fixed config (identical for every run below and across all 3 datasets):")
for k, v in BASE_PARAMS.items(): print(f"  {k:16s} {v}")

# Step 6 (notebook 07) reference points, transcribed from that notebook's own printed output.
REF = {
    "HI": {"V1": 0.7178, "v4 mean-pool + PLR (WIDE head, notebook 07)": 0.7245,
           "our XGBoost (tuned)": 0.7255, "published CatBoost": 0.726, "published MLP-PLR": 0.728},
    "FB": {"V1": 6.0379, "v4 mean-pool + PLR (WIDE head, notebook 07)": 5.7932,
           "our XGBoost (tuned)": 5.4016, "published CatBoost": 5.324, "published MLP-PLR": 5.525},
    "SA": {"V1": 0.9140, "v4 mean-pool + PLR (WIDE head, notebook 07)": 0.9227,
           "our XGBoost (tuned)": 0.9235, "published CatBoost": 0.923, "published MLP-PLR": 0.924},
}
for k in KEYS:
    print(f"\n{BENCHMARKS[k]['label']} reference points ({'higher better' if higher_is_better(k) else 'lower better'}):")
    for name, v in REF[k].items(): print(f"  {name:42s} {v:.4f}")

## 2. The sweep -- pooling-point baseline + NCL at 6 lambda values, x 3 datasets

In [ ]:
def _score(key, r):
    if DATA[key].task == "binary":
        return r["test_acc"]
    # benchmarks.py standardises regression targets before training; run_config's rmse is
    # therefore in STANDARDIZED units. Rescale to ORIGINAL units (same convention as
    # notebook 07/08's score_from_raw: rmse_original = rmse_standardized * y_std) so this
    # is directly comparable to every reference point above.
    return r["test_rmse"] * DATA[key].y_std

jobs = [(k, "meanpool", "embedding", 0.0, s) for k in KEYS for s in SEEDS] + \
       [(k, "meanpool_ncl", "output", lam, s) for k in KEYS for lam in LAMBDAS for s in SEEDS]
jobs.sort(key=lambda j: {"HI": 0, "SA": 1, "FB": 2}.get(j[0], 9))   # cheapest datasets first

t0 = time.time()
def _run(key, arm, pool_level, lam, seed):
    params = {**BASE_PARAMS}
    if arm == "meanpool_ncl":
        params["ncl_lambda"] = lam
    d = DATA[key]
    r = run_config(arm, params, ARRAYS[key], seed, task=d.task, output_dim=d.output_dim,
                   epochs=EPOCHS, patience=PATIENCE, threads=THREADS_PER_JOB)
    r["key"], r["pool_level"], r["ncl_lambda"] = key, pool_level, lam
    r["score"] = _score(key, r)
    return r

RESULTS = []
for i, r in enumerate(Parallel(n_jobs=N_JOBS, return_as="generator")(
        delayed(_run)(k, arm, pool_level, lam, s) for k, arm, pool_level, lam, s in jobs)):
    RESULTS.append(r)
    log(f"  [{i+1:3d}/{len(jobs)}] {BENCHMARKS[r['key']]['label']:18s} pool_level={r['pool_level']:9s} "
        f"lambda={r['ncl_lambda']:<4.1f} score={r['score']:.4f}  ({r['seconds']/60:.1f} min)  "
        f"elapsed {(time.time()-t0)/60:.1f} min")
RESULTS = pd.DataFrame(RESULTS)
log(f"all {len(jobs)} runs done in {(time.time()-t0)/60:.1f} min")

## 3. Score vs. lambda, per dataset

In [ ]:
SUMMARY = (RESULTS.groupby(["key", "pool_level", "ncl_lambda"])["score"].agg(["mean", "std"])
          .rename(columns={"mean": "score", "std": "sd"}).reset_index())

fig, axes = plt.subplots(1, len(KEYS), figsize=(6 * len(KEYS), 4.6))
axes = [axes] if len(KEYS) == 1 else list(axes)
BEST = {}
for ax, k in zip(axes, KEYS):
    s = SUMMARY[SUMMARY.key == k]
    base_row = s[s.pool_level == "embedding"].iloc[0]
    ncl_rows = s[s.pool_level == "output"].sort_values("ncl_lambda")
    hib = higher_is_better(k)
    best = ncl_rows.loc[ncl_rows["score"].idxmax() if hib else ncl_rows["score"].idxmin()]
    BEST[k] = (base_row, best, ncl_rows)

    ax.errorbar(ncl_rows["ncl_lambda"], ncl_rows["score"], yerr=ncl_rows["sd"].fillna(0.0),
               marker="o", color="#9c1c47", capsize=4, label="meanpool_ncl")
    ax.axhline(base_row["score"], color="#7b5cff", linestyle="--",
              label=f"embedding-level baseline: {base_row['score']:.4f}")
    ax.axhline(REF[k]["our XGBoost (tuned)"], color="#ff8a3d", linestyle=":",
              label=f"our XGBoost: {REF[k]['our XGBoost (tuned)']:.4f}")
    ax.set_xlabel("ncl_lambda"); ax.set_ylabel(f"test {'acc' if hib else 'rmse'}")
    ax.set_title(f"{BENCHMARKS[k]['label']} ({'higher' if hib else 'lower'} better)", fontweight="bold")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for k in KEYS:
    base_row, best, _ = BEST[k]
    rows2 = [{"model": "embedding-level baseline (today's v4)", "score": base_row["score"], "sd": base_row["sd"]},
            {"model": f"meanpool_ncl, BEST lambda={best['ncl_lambda']:.1f}", "score": best["score"], "sd": best["sd"]},
            {"model": "our XGBoost (tuned)", "score": REF[k]["our XGBoost (tuned)"], "sd": np.nan},
            {"model": "v4 mean-pool + PLR (WIDE head, notebook 07)",
             "score": REF[k]["v4 mean-pool + PLR (WIDE head, notebook 07)"], "sd": np.nan},
            {"model": "published CatBoost", "score": REF[k]["published CatBoost"], "sd": np.nan},
            {"model": "published MLP-PLR", "score": REF[k]["published MLP-PLR"], "sd": np.nan}]
    print(f"\n{BENCHMARKS[k]['label']}:")
    show(pd.DataFrame(rows2).set_index("model").sort_values("score", ascending=not higher_is_better(k)),
        {"score": "{:.4f}", "sd": "{:.4f}"})

## 4. Findings

*(Computed from the run above.)*

In [ ]:
sep = "=" * 92
print(sep); print("FINDINGS -- does Negative Correlation Learning generalize beyond California Housing?"); print(sep)

verdicts = {}
for k in KEYS:
    base_row, best, ncl_rows = BEST[k]
    hib = higher_is_better(k)
    base_s, base_sd = base_row["score"], (base_row["sd"] if pd.notna(base_row["sd"]) else 0.0)
    best_s, best_sd = best["score"], (best["sd"] if pd.notna(best["sd"]) else 0.0)
    noise = max(base_sd, best_sd, 1e-9)
    delta = best_s - base_s
    beats_baseline = (delta > noise) if hib else (delta < -noise)
    beats_xgb = (best_s > REF[k]["our XGBoost (tuned)"]) if hib else (best_s < REF[k]["our XGBoost (tuned)"])
    verdicts[k] = beats_baseline

    print(f"\n{BENCHMARKS[k]['label']}:")
    print(f"   embedding-level baseline : {base_s:.4f} +/- {base_sd:.4f}")
    print(f"   best NCL (lambda={best['ncl_lambda']:.1f})    : {best_s:.4f} +/- {best_sd:.4f}   delta={delta:+.4f}")
    print(f"   beats baseline beyond seed noise : {'YES' if beats_baseline else 'no'}")
    print(f"   beats our tuned XGBoost           : {'YES' if beats_xgb else 'no'} ({REF[k]['our XGBoost (tuned)']:.4f})")

n_helped = sum(verdicts.values())
print(f"\nVerdict: NCL beat the embedding-level baseline on {n_helped} of {len(KEYS)} datasets tested here.")
if n_helped == len(KEYS):
    print("Generalizes cleanly across every dataset tested so far (California Housing + these "
         f"{len(KEYS)}). Strong case for building multiclass support next, to finally test it on "
         "Covertype -- the dataset with the actual unresolved gap.")
elif n_helped > 0:
    print("Mixed: helps on some datasets, not others. Worth checking whether the SAME lambda value "
         "wins across datasets, or whether it needs to be tuned per dataset (like everything else "
         "in this project, per Step 8's finding).")
else:
    print("Did not clearly help on any of these 3 -- the California Housing result may not "
         "generalize as-is. Worth checking whether a joint lambda x architecture search "
         "(rather than lambda alone, at a FIXED architecture) changes this.")
print(sep)

## 5. Notes / next steps

* Multiclass is still not supported (`ncl.py`'s penalty is only defined for a scalar
  prediction) -- Covertype, the dataset that actually needs a fix, still can't be tested.
* This is still ONE fixed architecture (notebook 06/07's) with only lambda varied, not a
  joint search -- same caveat as notebook 11.
* If this generalizes, the natural next steps are: (a) build multiclass support and test on
  Covertype, (b) a joint Optuna search over lambda + the rest of the architecture, the way
  Step 8 did for plain mean-pool.